# ThinkGuard 4단계: bge-m3 판단 모델 학습 (Kaggle GPU)

## 준비 (한 번만)
1. **데이터셋 만들기**: Kaggle → Datasets → New Dataset → `kaggle_upload/thinkguard_data.zip`을 올리고 이름을 `thinkguard-data`로 정한다. 공개 범위는 **Private**으로 둔다. zip은 자동으로 풀린다.
2. **노트북 만들기**: Code → New Notebook → File → Import Notebook으로 이 파일을 올린다.
3. 오른쪽 패널에서:
   - **Add Input** → Your Datasets → `thinkguard-data` 추가
   - **Settings → Accelerator → GPU T4 x2** (P100은 최신 PyTorch에서 지원이 빠져서 T4를 쓴다)
   - **Settings → Internet → On** (bge-m3 다운로드에 필요. 켜려면 Kaggle 계정에 휴대폰 인증이 되어 있어야 한다)

## 실행
- **추천: Save Version → Save & Run All (Commit)**. 브라우저를 닫아도 뒤에서 끝까지 돈다 (최대 12시간). 끝나면 Output 탭에서 결과를 받는다.
- 셀을 하나씩 돌려도 되지만, 대화형 세션은 브라우저를 오래 비워두면 끊긴다.
- 예상 시간 (T4, 3 epoch): 본 학습 2~3시간 + 비교 실험 1~2시간. Kaggle GPU는 주당 약 30시간까지 무료다.

## 0. GPU 확인 (Tesla T4가 보여야 함)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. 데이터 위치 찾기
`/kaggle/input` 아래에서 스크립트를 찾아 경로를 잡는다. 입력 폴더는 읽기 전용이라 스크립트는 작업 폴더로 복사한다.

In [ ]:
import glob
import os
import shutil

INPUT_ROOT = '/kaggle/input'
WORK = '/kaggle/working'
hits = glob.glob(f'{INPUT_ROOT}/**/scripts/04_train_bge_m3.py', recursive=True)
assert hits, 'thinkguard-data 데이터셋을 찾지 못했다. 오른쪽 패널 Add Input에서 추가했는지 확인.'
SRC = os.path.dirname(os.path.dirname(hits[0]))
JOINED, KO = f'{SRC}/data/joined', f'{SRC}/data/ko'
SCRIPTS = f'{WORK}/scripts'
shutil.copytree(f'{SRC}/scripts', SCRIPTS, dirs_exist_ok=True)
RESULTS = f'{WORK}/results'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # T4 x2 중 하나만 쓴다 (스크립트가 단일 GPU용)
print('데이터 위치:', SRC)
!wc -l {JOINED}/*.jsonl {KO}/*.jsonl

## 2. 패키지
Kaggle에 transformers가 이미 있지만 스크립트가 확인한 버전대(4.45 이상, 5 미만)로 맞춘다.

In [ ]:
!pip install -q "transformers>=4.45,<5" sentencepiece
import torch
import transformers

print('transformers', transformers.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 3. 코드 점검 (1~2분)
작은 모델로 64행만 학습해서 오류가 없는지 본다. 점수는 의미 없다.

In [ ]:
!python {SCRIPTS}/04_train_bge_m3.py --train-dirs {JOINED} {KO} --model intfloat/multilingual-e5-small --limit 64 --epochs 1 --max-len 128 --out-dir {WORK}/smoke

## 4. 본 학습: 영어 + 한국어, 현재 발화 + 이전 대화
- `--grad-checkpointing`: T4 메모리(15GB)에 맞추려고 켠다.
- `--save-model`: 가장 좋은 epoch의 모델을 저장한다 (약 2.3GB, `results/bge_m3/en_ko_full/best_model/`).
- test 점수가 영어(en)와 한국어(ko)로 나뉘어 출력된다.
- 메모리 부족(CUDA out of memory)이 나면 `--batch-size 4 --grad-accum 8`로 바꾼다.

In [ ]:
!python {SCRIPTS}/04_train_bge_m3.py --train-dirs {JOINED} {KO} --view full --max-len 512 --batch-size 8 --grad-accum 4 --epochs 3 --grad-checkpointing --save-model --out-dir {RESULTS}/bge_m3 --tag en_ko_full

## 5. (비교 실험) 현재 발화만 입력
TF-IDF에서는 현재 발화만 넣는 쪽이 더 좋았다. bge-m3에서도 그런지 비교한다. 모델은 저장하지 않는다. 시간이 부족하면 이 셀은 건너뛴다.

In [ ]:
!python {SCRIPTS}/04_train_bge_m3.py --train-dirs {JOINED} {KO} --view message --max-len 512 --batch-size 8 --grad-accum 4 --epochs 3 --grad-checkpointing --out-dir {RESULTS}/bge_m3 --tag en_ko_message

## 6. TF-IDF 기준선 (같은 데이터, CPU로 몇 분)
test를 영어와 한국어로 나눠서 따로 잰다.

In [ ]:
!python {SCRIPTS}/03_baseline_tfidf.py --data-dirs {JOINED} {KO} --test-dirs {JOINED} --views message --tag en_ko__test_en --out-dir {RESULTS}/baseline_tfidf
!python {SCRIPTS}/03_baseline_tfidf.py --data-dirs {JOINED} {KO} --test-dirs {KO} --views message --tag en_ko__test_ko --out-dir {RESULTS}/baseline_tfidf

## 7. 결과 비교

In [ ]:
import json


def row(name, r):
    print(f"{name:28s} request F1 {r['requests_complete_answer']['f1']:.3f} | attempt F1 {r['has_self_attempt_evidence']['f1']:.3f} | "
          f"learning=false F1 {r['is_learning_context']['f1_false_class']:.3f} | false_unlock {r['false_unlock']['count']}/{r['false_unlock']['of']}")

for lang, tag in (('en', 'en_ko__test_en'), ('ko', 'en_ko__test_ko')):
    p = f'{RESULTS}/baseline_tfidf/{tag}/metrics.json'
    if os.path.exists(p):
        row(f'tfidf_message test[{lang}]', json.load(open(p))['message']['test'])
for p in sorted(glob.glob(f'{RESULTS}/bge_m3/*/metrics.json')):
    m = json.load(open(p))
    for lang, r in m['test'].items():
        row(f"{p.split('/')[-2]} test[{lang}]", r)

## 8. 결과 내려받기용 묶음
모델을 뺀 지표·예측 파일만 `results_metrics.zip`으로 묶는다 (작음). 모델은 Output 탭의 `results/bge_m3/en_ko_full/best_model/`에서 따로 받는다.

In [ ]:
!cd {WORK} && zip -qr results_metrics.zip results -x "*/best_model/*" && ls -lh {WORK}/results_metrics.zip